# 03 — Relationships
**Covers rubric steps 5–6.** Compare pay across groups, then check whether differences persist within experience bands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("../data/processed/us_fulltime_clean.csv")
print(df.shape)
df.head()

In [ ]:
def median_table(by):
    return (df.groupby(by, observed=True)["pay"]
              .agg(n="count", median="median",
                   q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
              .round(0))

## 1. Pay by experience band

In [ ]:
display(median_table("exp_band"))
order = ["0-2", "3-5", "6-10", "11-20", "20+"]
ax = sns.boxplot(data=df, x="exp_band", y="pay", order=order, showfliers=False)
ax.set(xlabel="Years of experience", ylabel="Annual compensation (USD)")
plt.savefig("../figures/pay_by_experience.png", dpi=150, bbox_inches="tight"); plt.show()

## 2. Pay by role

In [ ]:
t = median_table("role").sort_values("median", ascending=False)
display(t)
ax = sns.boxplot(data=df, y="role", x="pay", order=t.index, showfliers=False)
ax.set(xlabel="Annual compensation (USD)", ylabel="")
plt.savefig("../figures/pay_by_role.png", dpi=150, bbox_inches="tight"); plt.show()

## 3. Pay by work arrangement

In [ ]:
display(median_table("RemoteWork"))
ax = sns.boxplot(data=df, x="RemoteWork", y="pay", showfliers=False)
ax.set(xlabel="", ylabel="Annual compensation (USD)"); plt.xticks(rotation=15)
plt.savefig("../figures/pay_by_remote.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Key check: does the work-arrangement gap hold within experience bands?
If remote workers are more senior, the overall gap could just reflect experience.

In [ ]:
pd.crosstab(df["exp_band"], df["RemoteWork"], normalize="index").round(2)  # who works remote, by experience

In [ ]:
pd.pivot_table(df, values="pay", index="exp_band", columns="RemoteWork",
               aggfunc=["median", "count"], observed=True).round(0)

In [ ]:
ax = sns.pointplot(data=df, x="exp_band", y="pay", hue="RemoteWork",
                   order=order, estimator=np.median, errorbar=None, dodge=0.3)
ax.set(xlabel="Years of experience", ylabel="Median compensation (USD)")
plt.savefig("../figures/pay_remote_within_experience.png", dpi=150, bbox_inches="tight"); plt.show()

**TODO:** Interpret — does the gap shrink, persist, or reverse? Flag cells with small n.

## 5. Same check for role × experience

In [ ]:
top_roles = df["role"].value_counts().head(5).index
pd.pivot_table(df[df["role"].isin(top_roles)], values="pay", index="role",
               columns="exp_band", aggfunc="median", observed=True).round(0)

## 6. Experience vs. pay

In [ ]:
ax = sns.scatterplot(data=df, x="WorkExp_num", y="log_pay", hue="RemoteWork", alpha=0.3, s=12)
ax.set(xlabel="Years of experience", ylabel="log(Annual compensation)")
plt.savefig("../figures/exp_vs_logpay.png", dpi=150, bbox_inches="tight"); plt.show()

## Interpretation and limitations
**TODO:** Summarize findings. Associations only — not causal. Note self-selection and missing-pay bias.